# 04 – Gold Data Warehouse (Task D02 · D03)

**CommerceFlow 360** · Mục báo cáo: 3.5, 3.6

Xây dựng Star Schema tầng Gold bằng dbt từ các bảng `silver_*`:

| Task | Model | Grain |
|---|---|---|
| D02 | `dim_customer`, `dim_product`, `dim_seller`, `dim_date`, `dim_geography` | 1 dòng / khóa tự nhiên (customer_unique_id, product_id, seller_id, ngày, ZIP prefix) |
| D03 | `fact_orders`, `fact_order_items`, `fact_payments`, `fact_reviews` | xem `docs/star_schema/bus_matrix.md` |

**Đầu vào:** các bảng `silver_*` do `03_Silver_Transformation.ipynb` tạo. Kiểm thử Gold (D04) chạy ở `05_Data_Quality.ipynb`.

## 1. Chuẩn bị môi trường

In [1]:
# Trên Colab: clone repository và cài thư viện (bỏ qua khi chạy trong repository trên máy cá nhân)
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not Path('/content/CommerceFlow360').exists():
        !git clone -q https://github.com/huyenpiano1711/CommerceFlow360.git /content/CommerceFlow360
    %cd /content/CommerceFlow360
    %pip install -q -r requirements.txt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'dbt_project' / 'dbt_project.yml').is_file())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.utils.project import DUCKDB_PATH, query, run_dbt

assert DUCKDB_PATH.is_file(), 'Chưa có commerceflow.duckdb - chạy notebook 02 và 03 trước.'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DUCKDB_PATH :', DUCKDB_PATH)

PROJECT_ROOT: D:\CommerceFlow360
DUCKDB_PATH : D:\CommerceFlow360\commerceflow.duckdb


## 2. D02 – Năm dimension

`dbt run` (không kèm test): các test Gold cần cả dimension lẫn fact nên chạy ở notebook 05.

In [2]:
_ = run_dbt('run', '--select', 'gold.dimensions')

$ dbt run --select gold.dimensions


16:52:25  Running with dbt=1.12.6
16:52:27  Registered adapter: duckdb=1.11.1
16:52:28  Found 22 models, 89 data tests, 10 sources, 522 macros
16:52:29  
16:52:29  Concurrency: 4 threads (target='dev')
16:52:29  
16:52:29  4 of 5 START sql table model main.dim_product .................................. [RUN]
16:52:29  3 of 5 START sql table model main.dim_geography ................................ [RUN]
16:52:29  1 of 5 START sql table model main.dim_customer ................................. [RUN]
16:52:29  2 of 5 START sql table model main.dim_date ..................................... [RUN]
16:52:30  2 of 5 OK created sql table model main.dim_date ................................ [OK in 0.92s]
16:52:30  5 of 5 START sql table model main.dim_seller ................................... [RUN]
16:52:30  4 of 5 OK created sql table model main.dim_product ............................. [OK in 1.13s]
16:52:30  3 of 5 OK created sql table model main.dim_geography ........................... [

In [3]:
DIMENSIONS = ['dim_customer', 'dim_product', 'dim_seller', 'dim_date', 'dim_geography']
dimension_counts = pd.concat([query(f"select '{t}' as table_name, count(*) as row_count from {t}") for t in DIMENSIONS])
display(dimension_counts)
assert dimension_counts['row_count'].gt(0).all(), 'Có dimension chưa có dữ liệu.'

,table_name,row_count
0,dim_customer,96096
0,dim_product,32951
0,dim_seller,3095
0,dim_date,1827
0,dim_geography,19177


## 3. D03 – Bốn fact

In [4]:
_ = run_dbt('run', '--select', 'gold.facts')

$ dbt run --select gold.facts


16:53:26  Running with dbt=1.12.6
16:53:29  Registered adapter: duckdb=1.11.1
16:53:32  Found 22 models, 89 data tests, 10 sources, 522 macros
16:53:32  
16:53:32  Concurrency: 4 threads (target='dev')
16:53:32  
16:53:33  3 of 4 START sql table model main.fact_payments ................................ [RUN]
16:53:33  4 of 4 START sql table model main.fact_reviews ................................. [RUN]
16:53:33  2 of 4 START sql table model main.fact_orders .................................. [RUN]
16:53:33  1 of 4 START sql table model main.fact_order_items ............................. [RUN]
16:53:35  4 of 4 OK created sql table model main.fact_reviews ............................ [OK in 2.22s]
16:53:35  3 of 4 OK created sql table model main.fact_payments ........................... [OK in 2.36s]
16:53:35  2 of 4 OK created sql table model main.fact_orders ............................. [OK in 2.50s]
16:53:35  1 of 4 OK created sql table model main.fact_order_items ..................

In [5]:
FACTS = ['fact_orders', 'fact_order_items', 'fact_payments', 'fact_reviews']
fact_counts = pd.concat([query(f"select '{t}' as table_name, count(*) as row_count from {t}") for t in FACTS])
display(fact_counts)
assert fact_counts['row_count'].gt(0).all(), 'Có fact chưa có dữ liệu.'

,table_name,row_count
0,fact_orders,99441
0,fact_order_items,112650
0,fact_payments,103886
0,fact_reviews,99224
